In [1]:
#!/usr/bin/env python3
"""
Twain volleyball -> ICS

Pulls a public Google Sheet, cleans it, filters for Mark Twain White Team,
tags home/away by gym name, and writes a calendar .ics.
"""

from __future__ import annotations
import argparse, io, re, hashlib
from datetime import timedelta
from typing import Iterable
import pandas as pd
import requests
from zoneinfo import ZoneInfo

# Defaults tuned to your sheet
DEFAULT_SHEET_ID = "1cPfgjhkCUEMBhPg-9WJJ_BWUTkYjrFcv"
DEFAULT_GID = "796482645"
TEAM_NAME = "Mark Twain White Team"
HOME_GYM = "Mark Twain Middle School"

# --- fetch ---------------------------------------------------------------

def fetch_sheet_csv(sheet_id: str, gid: str) -> pd.DataFrame:
    """Grab the live CSV export for a specific sheet gid."""
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

def read_xlsx(path: str) -> pd.DataFrame:
    """Local escape hatch if they change access."""
    return pd.read_excel(path, sheet_name=0)

# --- cleaning ------------------------------------------------------------

def std_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase, trim, compress spaces; drop empty rows/cols."""
    df = df.rename(columns=lambda c: re.sub(r"\s+", " ", str(c).strip()).lower())
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    return df

def first_existing(df: pd.DataFrame, names: Iterable[str]) -> str | None:
    for n in names:
        if n in df.columns:
            return n
    return None

def coalesce_series(df: pd.DataFrame, names: Iterable[str]) -> pd.Series:
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([pd.NA] * len(df))

def split_teams(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create team_a, team_b from either:
    - a single 'teams'/'matchup' col like 'A vs B' or 'A @ B'
    - paired columns like 'home team' + 'away team'
    """
    if "teams" in df.columns:
        raw = df["teams"].astype(str)
    else:
        t1 = coalesce_series(df, ["team 1", "team1", "home team", "home"])
        t2 = coalesce_series(df, ["team 2", "team2", "away team", "away", "opponent"])
        if t1.notna().any() and t2.notna().any():
            raw = t1.fillna("") + " vs " + t2.fillna("")
        else:
            raw = coalesce_series(df, ["matchup", "game", "event"]).astype(str)

    # Split on common separators
    parts = raw.str.split(r"\s*(?:vs\.?|v\.?|@|at|–|—|-|—|\u2013|\u2014)\s*",
                          n=1, regex=True, expand=True)
    df["team_a"] = parts[0].fillna("").str.strip()
    df["team_b"] = parts[1].fillna("").str.strip()
    return df

def parse_start_times(df: pd.DataFrame, tz: str) -> pd.Series:
    """Combine date + time columns into tz-aware datetimes."""
    date_col = first_existing(df, ["date", "game date"])
    time_col = first_existing(df, ["time", "start", "start time"])
    if not date_col:
        raise ValueError("No date-like column found")

    # Build a combined string and let pandas infer
    if time_col:
        combo = df[date_col].astype(str).str.strip() + " " + df[time_col].astype(str).str.strip()
    else:
        # no time? default 15:00
        combo = df[date_col].astype(str).str.strip() + " 15:00"

    dt = pd.to_datetime(combo, errors="coerce", infer_datetime_format=True)
    # Localize into the requested zone
    dt = dt.dt.tz_localize(ZoneInfo(tz), nonexistent="shift_forward", ambiguous="NaT")
    return dt

def home_away_from_location(loc: pd.Series, home_gym: str) -> pd.Series:
    """Home if gym string mentions the school, else away."""
    s = loc.fillna("").str.lower()
    home = s.str.contains(home_gym.lower(), na=False)
    return home.map({True: "home", False: "away"})

def ic_escape(s: str) -> str:
    """ICS escaping for commas, semicolons, newlines, and backslashes."""
    return (s or "").replace("\\", "\\\\").replace(";", "\\;").replace(",", "\\,").replace("\n", "\\n")

def make_uid(start, a, b, loc) -> str:
    key = f"{start}@{a}|{b}|{loc}".encode()
    return "mtvb-" + hashlib.md5(key).hexdigest() + "@local"

def to_ics(events: list[dict], tz: str, duration_min: int) -> str:
    lines = [
        "BEGIN:VCALENDAR",
        "VERSION:2.0",
        "PRODID:-//Twain VB//White Team Calendar//EN",
        "CALSCALE:GREGORIAN",
        "METHOD:PUBLISH",
        f"X-WR-TIMEZONE:{tz}",
    ]
    for ev in events:
        start = ev["start"]
        end = start + timedelta(minutes=duration_min)
        dtfmt = "%Y%m%dT%H%M%S"
        lines += [
            "BEGIN:VEVENT",
            f"UID:{make_uid(start, ev['team_a'], ev['team_b'], ev['location'])}",
            f"DTSTAMP:{start.astimezone(ZoneInfo('UTC')).strftime('%Y%m%dT%H%M%SZ')}",
            f"DTSTART;TZID={tz}:{start.strftime(dtfmt)}",
            f"DTEND;TZID={tz}:{end.strftime(dtfmt)}",
            f"SUMMARY:{ic_escape(ev['summary'])}",
            f"LOCATION:{ic_escape(ev['location'])}",
            f"DESCRIPTION:{ic_escape(ev['description'])}",
            "END:VEVENT",
        ]
    lines.append("END:VCALENDAR")
    return "\n".join(lines)

# --- pipeline ------------------------------------------------------------

def build_events(df: pd.DataFrame, tz: str, duration_min: int) -> list[dict]:
    df = std_cols(df)
    df = split_teams(df)

    # pick a location column by best guess
    loc_col = first_existing(df, ["location", "site", "venue", "gym", "place"]) or "location"
    if loc_col not in df.columns:
        df[loc_col] = ""

    # datetimes
    start = parse_start_times(df, tz)
    df["start"] = start

    # keep only rows with Twain White in either slot
    mask_white = (
        df["team_a"].str.contains(TEAM_NAME, case=False, na=False) |
        df["team_b"].str.contains(TEAM_NAME, case=False, na=False)
    )
    df = df[mask_white].copy()

    # tag home/away by gym name
    df["home_away"] = home_away_from_location(df[loc_col], HOME_GYM)

    # choose opponent and summary
    def opponent(row) -> tuple[str, str]:
        a, b = row["team_a"], row["team_b"]
        if re.search(TEAM_NAME, a, re.I):
            opp = b or "Opponent TBA"
            label = f"Mark Twain White vs {opp}"
        else:
            opp = a or "Opponent TBA"
            label = f"Mark Twain White at {opp}"
        # add a quick suffix for context
        label = f"Volleyball: {label} ({row['home_away']})"
        return opp, label

    opp_sum = df.apply(opponent, axis=1, result_type="expand")
    df["opponent"] = opp_sum[0]
    df["summary"] = opp_sum[1]
    df["location"] = df[loc_col].fillna("")

    # drop any rows missing a start time
    df = df[df["start"].notna()].sort_values("start")

    # pack events
    events = []
    for _, r in df.iterrows():
        desc = f"{r['team_a']} vs {r['team_b']}\nSource: Google Sheet"
        events.append({
            "start": r["start"],
            "team_a": r["team_a"],
            "team_b": r["team_b"],
            "location": r["location"],
            "summary": r["summary"],
            "description": desc,
        })
    return events

# --- cli ----------------------------------------------------------------

def main():
    p = argparse.ArgumentParser(description="Create an ICS for Mark Twain White Team games")
    p.add_argument("--sheet-id", default=DEFAULT_SHEET_ID, help="Google Sheet ID")
    p.add_argument("--gid", default=DEFAULT_GID, help="Sheet gid")
    p.add_argument("--xlsx", help="Optional local Excel file instead of Google")
    p.add_argument("--out", required=True, help="Output .ics path")
    p.add_argument("--tz", default="America/Los_Angeles", help="IANA timezone")
    p.add_argument("--duration-min", type=int, default=60, help="Event length in minutes")
    args = p.parse_args()

    # Fetch
    if args.xlsx:
        df = read_xlsx(args.xlsx)
    else:
        df = fetch_sheet_csv(args.sheet_id, args.gid)

    # Build and write
    events = build_events(df, tz=args.tz, duration_min=args.duration_min)
    cal = to_ics(events, tz=args.tz, duration_min=args.duration_min)
    with open(args.out, "w", encoding="utf-8") as f:
        f.write(cal)

    print(f"Wrote {len(events)} events to {args.out}")

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--sheet-id SHEET_ID] [--gid GID]
                             [--xlsx XLSX] --out OUT [--tz TZ]
                             [--duration-min DURATION_MIN]
ipykernel_launcher.py: error: the following arguments are required: --out


SystemExit: 2

/Users/mstiles/github/notebooks/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
